# Comparison with ArcPy

This notebook compares the usage of `footbridge` with `arcpy`.

In [1]:
import os
from pathlib import Path

import arcpy
from util import Timer, get_dataset

import footbridge as ft

timer = Timer()

# large dataset of US National Highway System roads, one feature class
# https://hepgis-usdot.hub.arcgis.com/datasets/dce9f09392eb474c8ad8e6a78416279b_0
gdb_path = get_dataset()

arcpy.env.overwriteOutput = True

Done! The .gdb already exists


## Load feature class into memory
### ArcPy

In [2]:
timer.start()
# ------------
arcpy.env.workspace = gdb_path
arcpy_fcs = arcpy.ListFeatureClasses()
fc_name = arcpy_fcs[0]
fc_path = os.path.join(gdb_path, fc_name)
arcpy_fc = os.path.join("memory", fc_name)
arcpy.conversion.ExportFeatures(fc_path, arcpy_fc)
# -----------
timer.stop()

13.8083 seconds


### footbridge

In [3]:
timer.start()
# ------------
ft_fc = ft.GeoDatabase(gdb_path)[0]
# -----------
timer.stop()

8.4328 seconds


|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |        ❌       |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Count rows
### ArcPy

In [4]:
timer.start()
# ------------
count_result = arcpy.management.GetCount(arcpy_fc)
row_count = count_result[0]
# -----------
timer.stop()
f"{row_count} rows"

0.116 seconds


'492005 rows'

### footbridge

In [5]:
timer.start()
# ------------
row_count = len(ft_fc)
# -----------
timer.stop()
f"{row_count} rows"

0.0001 seconds


'492005 rows'

|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Filter rows
### ArcPy

In [6]:
timer.start()
# ------------
arcpy_fc_filtered = os.path.join("memory", fc_name + "_filtered")
arcpy.conversion.ExportFeatures(
    arcpy_fc, arcpy_fc_filtered, where_clause="ROUTEID = '50'"
)
# -----------
timer.stop()
f"{arcpy.management.GetCount(arcpy_fc_filtered)[0]} rows"

0.1374 seconds


'24 rows'

### footbridge

In [7]:
timer.start()
# ------------
ft_fc_filtered = ft_fc.select_rows("ROUTEID == '50'")
# -----------
timer.stop()
f"{len(ft_fc_filtered)} rows"

0.0162 seconds


'24 rows'

|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Get the last row
### ArcPy

In [8]:
timer.start()
# ------------
with arcpy.da.SearchCursor(arcpy_fc, field_names=["*"]) as s_cursor:
    for row in s_cursor:
        this_row = row
# -----------
timer.stop()
this_row

2.2269 seconds


(492005,
 (-84.7304625142813, 33.75313141928635),
 '2025.08.08',
 2022.0,
 13.0,
 97.0,
 '1097200063903INC',
 0.0,
 0.15,
 ' ',
 ' ',
 ' ',
 ' ',
 8.0,
 1.0,
 'GA48T',
 'GA48T_01',
 'From I-20 Exit 37 (SR 92/Fairburn Rd): west on SR 92 roughly 0.9mi to Hospital Drive, south on Hospi',
 1.91,
 ' ',
 7.0,
 2.0,
 0.0,
 0.0,
 4.0,
 3817.0,
 0.0,
 0.0,
 0.0,
 0.0,
 None,
 0.015218,
 datetime.datetime(2025, 8, 7, 0, 0),
 'Add (Ref Memo: HEPP-0725-NHSM-0004)',
 'GA_HPMS_FULL_2022')

### footbridge

In [9]:
timer.start()
# ------------
this_row = ft_fc[-1]
# -----------
timer.stop()
this_row

0.0019 seconds


,VERSION,YEAR,STFIPS,CTFIPS,ROUTEID,BEGINPOINT,ENDPOINT,SIGN1,SIGNT1,SIGNN1,...,AADT_COM,AADT_SINGL,FUT_AADT,FUT_YEAR,MILES,UPDATE_DAT,NHS_ACTION,FILE_NAME,SHAPE_Length,geometry
ObjectID,,,,,,,,,,,,,,,,,,,,,
492004,2025.08.08,2022.0,13.0,97.0,1097200063903INC,0.0,0.15,,,,...,0.0,0.0,0.0,NaT,0.015218,2025-08-07 00:00:00+00:00,Add (Ref Memo: HEPP-0725-NHSM-0004),GA_HPMS_FULL_2022,0.000245,"MULTILINESTRING ((-84.73056 33.75305, -84.73045 33.75314, -84.73037 33.75321))"


|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Get a row at an index

In [10]:
row_index = 400000

### ArcPy

In [11]:
timer.start()
# ------------
with arcpy.da.SearchCursor(arcpy_fc, field_names=["*"]) as s_cursor:
    for idx, row in enumerate(s_cursor):
        if idx == row_index:
            this_row = row
            break
# -----------
timer.stop()
this_row

1.8354 seconds


(400001,
 (-76.62334059749135, 40.190563605963455),
 '2025.08.08',
 2020.0,
 42.0,
 43.0,
 '4 06 ATEV - 12215',
 7.941,
 8.622,
 'S283',
 'S',
 '283',
 ' ',
 7.0,
 1.0,
 ' ',
 ' ',
 ' ',
 0.0,
 'P',
 7.0,
 2.0,
 2.0,
 0.0,
 0.0,
 99999.0,
 289.0,
 0.0,
 0.0,
 0.0,
 None,
 0.68232,
 None,
 ' ',
 'PA_NHS_2021')

### footbridge

In [12]:
timer.start()
# ------------
this_row = ft_fc[row_index]
# -----------
timer.stop()
this_row

0.0016 seconds


,VERSION,YEAR,STFIPS,CTFIPS,ROUTEID,BEGINPOINT,ENDPOINT,SIGN1,SIGNT1,SIGNN1,...,AADT_COM,AADT_SINGL,FUT_AADT,FUT_YEAR,MILES,UPDATE_DAT,NHS_ACTION,FILE_NAME,SHAPE_Length,geometry
ObjectID,,,,,,,,,,,,,,,,,,,,,
400000,2025.08.08,2020.0,42.0,43.0,4 06 ATEV - 12215,7.941,8.622,S283,S,283,...,0.0,0.0,0.0,NaT,0.68232,NaT,,PA_NHS_2021,0.012664,"MULTILINESTRING ((-76.62953 40.19192, -76.62901 40.19181, -76.62749 40.19148, -76.62478 40.19089, -76.62336 40.19058, -76.62233 40.19037, -76.62055 40.18998, -76.61972 40.18979, -76.61885 40.18957, -76.61795 40.18929, -76.6172 40.18904))"


|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Get a column

In [13]:
column_name = "MILES"

### ArcPy

In [14]:
timer.start()
# ------------
fc_field_mapping = arcpy.FieldMappings()
fc_field_mapping.addTable(arcpy_fc)
mapping_index = fc_field_mapping.findFieldMapIndex(column_name)
field_map = fc_field_mapping.getFieldMap(mapping_index)

new_field_mapping = arcpy.FieldMappings()
new_field_mapping.addFieldMap(field_map)

arcpy_fc_col = os.path.join("memory", fc_name + "_col")
result = arcpy.conversion.ExportFeatures(
    arcpy_fc, arcpy_fc_col, field_mapping=new_field_mapping
)
# -----------
timer.stop()
[field.name for field in arcpy.ListFields(arcpy_fc_col)]

3.7345 seconds


['OBJECTID', 'Shape', 'MILES']

### footbridge

In [15]:
timer.start()
# ------------
ft_fc_col = ft_fc.select_columns(column_name)
# -----------
timer.stop()
ft_fc_col.list_fields()

0.0602 seconds


['ObjectID', 'MILES', 'geometry']

|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Field calculator
### ArcPy

In [16]:
timer.start()
# ------------
arcpy.management.CalculateField(arcpy_fc, "KILOMETERS", "!MILES! * 1.60934", "PYTHON3")
# -----------
timer.stop()
[field.name for field in arcpy.ListFields(arcpy_fc)]

14.7647 seconds


['OBJECTID',
 'Shape',
 'VERSION',
 'YEAR',
 'STFIPS',
 'CTFIPS',
 'ROUTEID',
 'BEGINPOINT',
 'ENDPOINT',
 'SIGN1',
 'SIGNT1',
 'SIGNN1',
 'LNAME',
 'NHS',
 'STATUS',
 'FACID',
 'CONNID',
 'CONNDES',
 'CONNMILES',
 'ACLASS',
 'FCLASS',
 'FACILITYT',
 'THROUGH_LA',
 'SPEED_LIMI',
 'OWNERSHIP',
 'URBANCODE',
 'AADT',
 'AADT_COM',
 'AADT_SINGL',
 'FUT_AADT',
 'FUT_YEAR',
 'MILES',
 'UPDATE_DAT',
 'NHS_ACTION',
 'FILE_NAME',
 'KILOMETERS']

### footbridge

In [17]:
timer.start()
# ------------
ft_fc.calculate("KILOMETERS", "MILES * 1.60934")
# -----------
timer.stop()
ft_fc.list_fields()

0.0784 seconds


['ObjectID',
 'VERSION',
 'YEAR',
 'STFIPS',
 'CTFIPS',
 'ROUTEID',
 'BEGINPOINT',
 'ENDPOINT',
 'SIGN1',
 'SIGNT1',
 'SIGNN1',
 'LNAME',
 'NHS',
 'STATUS',
 'FACID',
 'CONNID',
 'CONNDES',
 'CONNMILES',
 'ACLASS',
 'FCLASS',
 'FACILITYT',
 'THROUGH_LA',
 'SPEED_LIMI',
 'OWNERSHIP',
 'URBANCODE',
 'AADT',
 'AADT_COM',
 'AADT_SINGL',
 'FUT_AADT',
 'FUT_YEAR',
 'MILES',
 'UPDATE_DAT',
 'NHS_ACTION',
 'FILE_NAME',
 'SHAPE_Length',
 'KILOMETERS',
 'geometry']

|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ✅      |       ✅       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Project
### ArcPy

In [18]:
timer.start()
# ------------
arcpy_fc_projected = os.path.join("memory", fc_name + "_projected")
out_crs = arcpy.SpatialReference(102008)  # North America Albers Equal Area Conic
arcpy.management.Project(arcpy_fc_filtered, arcpy_fc_projected, out_crs)
# -----------
timer.stop()

0.6153 seconds


## footbridge

In [19]:
timer.start()
# ------------
ft_fc_projected = ft.project(ft_fc_filtered, crs="ESRI:102008")
# -----------
timer.stop()

0.0327 seconds


|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ✅       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Buffer
### ArcPy

In [20]:
timer.start()
# ------------
arcpy_fc_buffered = os.path.join("memory", fc_name + "_buffered")
arcpy.analysis.PairwiseBuffer(arcpy_fc_projected, arcpy_fc_buffered, "10 Miles")
# -----------
timer.stop()

0.1564 seconds


### footbridge

In [21]:
timer.start()
# ------------
ft_fc_buffered = ft.buffer(ft_fc_projected, 10 * 1609.344)  # geometry units are meters
# -----------
timer.stop()

0.0041 seconds


|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ✅       |       ❌        |    ❌    |
| footbridge |      ✅      |       ✅       |       ✅        |    ✅    |

## Save to disk

In [22]:
out_folder = Path(".")

### ArcPy

In [23]:
timer.start()
# ------------
arcpy.management.CreateFileGDB(out_folder, "test1.gdb")
arcpy.conversion.ExportFeatures(arcpy_fc, out_folder / "test1.gdb" / "highways")
# -----------
timer.stop()

15.6228 seconds


### footbridge

In [24]:
timer.start()
# ------------
ft_fc.save(out_folder / "test2.gdb", "highways")
# -----------
timer.stop()

19.8791 seconds


|           | Fewer lines | More pythonic | Faster runtime | Overall |
|----------:|:-----------:|:-------------:|:--------------:|:-------:|
|     ArcPy |      ❌      |       ❌       |       ✅        |    ✅    |
| footbridge |      ✅      |       ✅       |       ❌        |    ✅    |

## Summary

|                       | ArcPy | footbridge |
|----------------------:|:-----:|:---------:|
|    Load feature class |   ❌   |     ✅     |
|            Count rows |   ❌   |     ✅     |
|           Filter rows |   ❌   |     ✅     |
|      Get the last row |   ❌   |     ✅     |
| Get a row at an index |   ❌   |     ✅     |
|          Get a column |   ❌   |     ✅     |
|      Field calculator |   ❌   |     ✅     |
|               Project |   ❌   |     ✅     |
|                Buffer |   ❌   |     ✅     |
|       Copy to new GDB |   ✅   |     ✅     |
